# 3D toric code — XZ-plane magnetic ($h_x$) transition: FM / S₂ / M / stabilizers across $L$ and $h_z$

Canonical notebook for the topological→trivial **magnetic** ($h_x$-driven) line, from the
fixed-$h_z$ hx-sweep trees `phase_hz{0.0…1.1}/L*`. Combines the four observables in one place:

| obs | source | behaviour across the transition |
|---|---|---|
| **O_FM** | `results/xz_line_fm/` (magnetic membrane, `fm.py`) | rises $0\to1$ — the topological **order parameter** |
| **S2** | `results/xz_line_s2/` (central-plaquette Rényi-2) | drops $3\ln2\to0$ — local entanglement locator |
| **M_x** | `results/xz_line_energy/` (⟨σ_x⟩) | rises $0\to1$ — smooth local magnetisation (UNDER-estimates $h_x^c$) |
| **B_p** | `results/xz_line_energy/` (⟨B_p⟩) | drops — plaquette stabiliser / topological order |
| (A_v, Vscore, E also available from the energy dumps) | | |

**Data is filled incrementally** as cluster jobs finish — the notebook shows whatever is present
(§2 prints the inventory). Use the **`SHOW_OBS` / `SHOW_L` / `SHOW_HZ`** knobs in §1 to plot only
what you want, not every panel at once.

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os, re
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
DIRS = {"fm": f"{ROOT}/xz_line_fm", "s2": f"{ROOT}/xz_line_s2", "energy": f"{ROOT}/xz_line_energy"}

# ---- THE PLOT-SELECTION KNOBS ----
SHOW_OBS = ["O_FM", "S2", "M_x", "B_p"]   # which observables to plot (subset of OBS below)
SHOW_L   = [4, 5, 6, 7]                    # which system sizes to include (present ones auto-filter)
SHOW_HZ  = None                           # list of hz to show, or None = all available

LOCATOR  = "S2"        # observable that defines the boundary / windows. S2 (local central-
                       # plaquette Renyi) is the more reliable locator at SMALL L, where the
                       # R=1 FM membrane is heavy-tailed; switch to "O_FM" for the large-L order param.
LOCATOR_FALLBACK = ["S2", "M_x", "B_p", "O_FM"]   # if LOCATOR is nan for a cut, try these in order
FM_KURT_MAX = 200.0    # mask O_FM points whose B3 excess-kurtosis exceeds this (heavy-tailed / unreliable)

# ---- window finalizer (§6) ----
PAD  = 0.25            # larger-L window = locator h_c ± PAD
NPTS = 7
WINDOWS_FINAL = {}     # {hz: (lo, hi)} manual overrides; leave a key out to accept the auto window

# observable registry: name -> (source_key, value_key, err_key, "up"/"down")
OBS = {
    "O_FM":   ("fm",     "O",      "Oe",   "up"),
    "S2":     ("s2",     "S2",     "S2e",  "down"),
    "M_x":    ("energy", "mx",     None,   "up"),
    "B_p":    ("energy", "B_p",    None,   "down"),
    "A_v":    ("energy", "A_v",    None,   "down"),
    "Vscore": ("energy", "Vscore", None,   "peak"),
    "E":      ("energy", "E",      None,   "mono"),
}
LABEL = {"O_FM": r"$O_{FM}^{m}$", "S2": r"$S_2$", "M_x": r"$\langle M_x\rangle$",
         "B_p": r"$\langle B_p\rangle$", "A_v": r"$\langle A_v\rangle$",
         "Vscore": "Vscore", "E": "$E_0$"}

## 2 · Data library

Globs the three dirs, parses `(L, h_z)` from each filename, and builds
`DATA[obs][(L, hz)] = {hx, y, ye, raw}`. The inventory table shows coverage — blanks are
combinations whose cluster job hasn't finished/extracted yet.

In [ ]:
# ====================== 2 · DATA LIBRARY ======================
_LHZ = re.compile(r"_L(\d+)_hz(\d+(?:\.\d+)?)")
def _parse(fn):
    m = _LHZ.search(fn); return (int(m.group(1)), round(float(m.group(2)), 4)) if m else (None, None)

def _load_source(src, val_key, err_key):
    recs = {}
    for jp in sorted(glob.glob(os.path.join(DIRS[src], "*.json"))):
        L, hz = _parse(os.path.basename(jp))
        if L is None: continue
        d = json.load(open(jp))
        hx = np.array(d.get("field", []), float)
        if hx.size == 0: continue
        o = np.argsort(hx)
        y = np.array(d.get(val_key, []), float)
        ye = np.array(d.get(err_key, np.zeros(len(hx))) if err_key else np.zeros(len(hx)), float)
        recs[(L, hz)] = dict(hx=hx[o], y=y[o] if y.size else y, ye=ye[o] if ye.size else ye, raw=d)
    return recs

DATA = {obs: _load_source(src, vk, ek) for obs, (src, vk, ek, _) in OBS.items()}

# inventory
all_L = sorted({L for obs in DATA.values() for (L, _) in obs})
all_hz = sorted({hz for obs in DATA.values() for (_, hz) in obs})
print("coverage  (rows=obs, cols=hz; digits = L present)\n")
hdr = "        " + "".join(f"{hz:>6}" for hz in all_hz)
print(hdr)
for obs in OBS:
    if not DATA[obs]: continue
    row = f"{obs:>7} "
    for hz in all_hz:
        Ls = "".join(str(L) for L in all_L if (L, hz) in DATA[obs])
        row += f"{Ls:>6}"
    print(row)
print(f"\nL present: {all_L}   hz present: {all_hz}")

## 3 · Observable curves vs $h_x$

One figure **per selected observable** (`SHOW_OBS`); within it, one panel per $L$, overlaying the
$h_z$ cuts (viridis). This is the "how does it change across $h_x$ and $h_z$, at each $L$" view.
O_FM points failing the B3 heavy-tail cut (`FM_KURT_MAX`) are drawn hollow.

In [ ]:
# ====================== 3 · CURVES vs hx (per selected observable) ======================
def hz_list(obs):
    hzs = sorted({hz for (L, hz) in DATA[obs] if (SHOW_HZ is None or hz in SHOW_HZ)})
    return hzs

def fm_mask(rec):
    """boolean mask of TRUSTWORTHY O_FM points (B3 kurtosis below FM_KURT_MAX)."""
    k = rec["raw"].get("b3_max_kurt")
    if k is None: return np.ones(len(rec["hx"]), bool)
    k = np.array(k, float)[np.argsort(np.array(rec["raw"]["field"], float))]
    return k <= FM_KURT_MAX

for obs in SHOW_OBS:
    if not DATA[obs]:
        print(f"[{obs}] no data yet — skipping"); continue
    Ls = [L for L in SHOW_L if any((L, hz) in DATA[obs] for hz in all_hz)]
    if not Ls: continue
    hzs = hz_list(obs); norm = plt.Normalize(min(hzs), max(hzs)); cmap = plt.cm.viridis
    fig, ax = plt.subplots(1, len(Ls), figsize=(4.6*len(Ls), 4.0), squeeze=False)
    for j, L in enumerate(Ls):
        a = ax[0, j]
        for hz in hzs:
            rec = DATA[obs].get((L, hz))
            if rec is None: continue
            c = cmap(norm(hz))
            a.errorbar(rec["hx"], rec["y"], yerr=rec["ye"], fmt="-o", ms=3, lw=1,
                       capsize=2, color=c, label=f"hz={hz}")
            if obs == "O_FM":
                bad = ~fm_mask(rec)
                if bad.any():
                    a.scatter(rec["hx"][bad], rec["y"][bad], s=55, facecolors="none",
                              edgecolors="red", lw=1.2, zorder=5)
        a.set(xlabel="$h_x$", ylabel=LABEL[obs], title=f"{LABEL[obs]}   L={L}")
        if obs == "S2": a.axhline(3*np.log(2), ls=":", c="grey", lw=0.8)
        if obs == "M_x": a.axhline(0.5, ls=":", c="grey", lw=0.8)
        if j == len(Ls)-1: a.legend(fontsize=7, ncol=2)
    fig.suptitle(f"{LABEL[obs]} vs $h_x$  (hollow red = B3-flagged)" if obs=="O_FM"
                 else f"{LABEL[obs]} vs $h_x$", y=1.02)
    plt.tight_layout(); plt.show()

## 4 · Transition points $h_x^c(L, h_z)$

Per $(L, h_z)$: **O_FM** uses the stored logistic $h_c$; **S2 / M_x / B_p** use the
parabola-refined extremum of $|d\,\text{obs}/dh_x|$ (heavy-tail-masked for O_FM). The table lets you
compare locators — O_FM is the order parameter; M_x sits systematically low.

In [ ]:
# ====================== 4 · TRANSITION POINTS ======================
def parab_peak(x, y):
    i = int(np.argmax(y))
    if i == 0 or i == len(y)-1: return np.nan
    x0,x1,x2 = x[i-1:i+2]; y0,y1,y2 = y[i-1:i+2]
    d = (x0-x1)*(x0-x2)*(x1-x2)
    if d == 0: return x1
    a = (x2*(y1-y0)+x1*(y0-y2)+x0*(y2-y1))/d
    b = (x2*x2*(y0-y1)+x1*x1*(y2-y0)+x0*x0*(y1-y2))/d
    return x1 if a == 0 else -b/(2*a)

def _cross(x, y, level):
    for i in range(len(y)-1):
        if (y[i]-level)*(y[i+1]-level) <= 0 and y[i+1] != y[i]:
            t = (level-y[i])/(y[i+1]-y[i]); return x[i]+t*(x[i+1]-x[i])
    return np.nan

def _fm_good(rec):
    """B3-trustworthy O_FM points (excess kurtosis <= FM_KURT_MAX)."""
    k = rec["raw"].get("b3_max_kurt")
    if k is None: return np.ones(len(rec["hx"]), bool)
    k = np.array(k, float)[np.argsort(np.array(rec["raw"]["field"], float))]
    return k <= FM_KURT_MAX

def hc_of(obs, L, hz):
    """h_x^c estimate. O_FM: B3-masked O=0.5 crossing (robust to heavy-tailed low-hx
    points; the stored logistic h_c is corrupted when those are noisy) -> fallback to
    stored h_c. Others: parabola-refined |d obs/dh_x| peak (steepest-change field)."""
    rec = DATA[obs].get((L, hz))
    if rec is None or rec["y"].size < 3: return np.nan
    hx, y = rec["hx"], rec["y"]
    if obs == "O_FM":
        m = _fm_good(rec)
        hc = _cross(hx[m], y[m], 0.5) if m.sum() >= 2 else np.nan
        if not np.isfinite(hc):
            hc0 = rec["raw"].get("h_c")
            hc = float(hc0) if hc0 is not None and np.isfinite(hc0) else np.nan
    else:
        hc = parab_peak(hx, np.abs(np.gradient(y, hx)))   # steepest-change field
    # a locator outside the swept window is meaningless (bad fit / no crossing) -> nan
    return hc if np.isfinite(hc) and hx.min()-1e-9 <= hc <= hx.max()+1e-9 else np.nan

LOCATORS = [o for o in ("O_FM", "S2", "M_x", "B_p") if DATA.get(o)]
pairs = sorted({(L, hz) for o in LOCATORS for (L, hz) in DATA[o]
                if L in SHOW_L and (SHOW_HZ is None or hz in SHOW_HZ)})
print(f"{'L':>2} {'hz':>5} " + "".join(f"{o:>8}" for o in LOCATORS))
HC = {}
for (L, hz) in pairs:
    vals = {o: hc_of(o, L, hz) for o in LOCATORS}
    HC[(L, hz)] = vals
    print(f"{L:>2} {hz:>5} " + "".join(
        (f"{vals[o]:>8.3f}" if np.isfinite(vals[o]) else f"{'-':>8}") for o in LOCATORS))

## 5 · Phase boundary $h_x^c$ vs $h_z$ (across $L$)

The `LOCATOR` observable's $h_x^c$ vs $h_z$, one series per $L$ — the XZ-plane boundary and its
finite-size drift. Overlaid faint markers show the other locators at each point for context.

In [ ]:
# ====================== 5 · BOUNDARY hx_c vs hz (per L) ======================
Ls = sorted({L for (L, _) in HC})
cmap = plt.cm.plasma; norm = plt.Normalize(min(Ls), max(Ls)+0.001)
fig, axb = plt.subplots(figsize=(7, 6))
for L in Ls:
    pts = sorted((hz, HC[(L, hz)].get(LOCATOR, np.nan)) for (Lx, hz) in HC if Lx == L)
    pts = [(hz, v) for hz, v in pts if np.isfinite(v)]
    if not pts: continue
    hz_a, hc_a = zip(*pts)
    axb.plot(hc_a, hz_a, "-o", color=cmap(norm(L)), label=f"L={L} ({LOCATOR})", zorder=3)
# faint: other locators (L=4 only, to avoid clutter)
for (L, hz), vals in HC.items():
    if L != min(Ls): continue
    for o, mk in [("M_x", "x"), ("S2", "+"), ("B_p", "1")]:
        if o != LOCATOR and np.isfinite(vals.get(o, np.nan)):
            axb.scatter(vals[o], hz, s=22, c="grey", marker=mk, alpha=0.5, zorder=1)
axb.set(xlabel=r"$h_x^c$", ylabel=r"$h_z$",
        title=f"XZ magnetic transition line — locator: {LOCATOR}")
axb.annotate("TOPOLOGICAL", (axb.get_xlim()[0], 0.9), color="navy", fontsize=9)
axb.annotate("TRIVIAL\n(x-polarized)", (axb.get_xlim()[1], 0.2), ha="right", color="darkred", fontsize=9)
axb.grid(alpha=0.3); axb.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6 · Finalize the L=5/6/7 windows

Auto window = `LOCATOR` $h_x^c \pm$ `PAD` (uses the best available $L$ per $h_z$, preferring the
largest). Override any cut in `WINDOWS_FINAL` (§1). Prints the hx grids and paste-ready
`HZ=… HXLO=… HXHI=… HXN=…` lines for `nersc/submit_nqs_hx_sweep.sh`.

In [ ]:
# ====================== 6 · WINDOW FINALIZER ======================
def best_hc(hz):
    """h_c at the largest L available for this hz, trying LOCATOR then the fallback chain
    (S2 -> M_x -> B_p -> O_FM by default) so no cut is left blank when one locator is nan."""
    for obs in dict.fromkeys([LOCATOR] + LOCATOR_FALLBACK):   # LOCATOR first, dedup, keep order
        cands = [(L, HC[(L, hz)].get(obs)) for (L, h) in HC if h == hz]
        cands = [(L, v) for L, v in cands if v is not None and np.isfinite(v)]
        if cands: return max(cands)[1]                        # largest L
    return np.nan

hzs = sorted({hz for (_, hz) in HC})
print(f"{'hz':>5} {'hc':>7} {'window':>18} {'grid':>44}")
for hz in hzs:
    if hz in WINDOWS_FINAL: lo, hi = WINDOWS_FINAL[hz]; src = "manual"
    else:
        hc = best_hc(hz); lo, hi = (round(hc-PAD,3), round(hc+PAD,3)) if np.isfinite(hc) else (np.nan, np.nan); src = "auto"
    grid = np.round(np.linspace(lo, hi, NPTS), 3) if np.isfinite(lo) else []
    print(f"{hz:>5} {best_hc(hz):>7.3f} {f'[{lo}, {hi}] ({src})':>18} {'  '.join(f'{g:g}' for g in grid):>44}")
print("\n# submit lines (one hx-sweep array per hz):")
for hz in hzs:
    if hz in WINDOWS_FINAL: lo, hi = WINDOWS_FINAL[hz]
    else:
        hc = best_hc(hz); lo, hi = (round(hc-PAD,3), round(hc+PAD,3)) if np.isfinite(hc) else (np.nan, np.nan)
    if np.isfinite(lo): print(f"HZ={hz} HXLO={lo} HXHI={hi} HXN={NPTS} L=<5|6|7> bash nersc/submit_nqs_hx_sweep.sh")

## 7 · Playground — pick cuts, see 4 observables vs $h_x$

Self-contained exploration cell (doesn't need the sections above). Edit `CUTS` to a list of `(L, hz)` pairs and re-run just this cell: each cut overlays as one line across **O_FM · S₂ · stabilizers (⟨B_p⟩ solid, ⟨A_v⟩ dashed) · ⟨M_x⟩** vs $h_x$.

In [ ]:
# ═══════════════ PLAYGROUND — 4 observables vs hx for chosen cuts ═══════════════
# Self-contained: edit CUTS and re-run THIS cell alone (no need to run the sections
# above). Each (L, hz) is overlaid as one coloured line in all four panels.
import glob, json, os, re
import numpy as np
import matplotlib.pyplot as plt

CUTS = [(4, 0.0), (4, 0.1), (4, 0.2), (4, 0.3), (4, 0.4), (4, 0.5), (4, 0.7), (4, 0.9), (4, 1.0), (4, 1.1)]     # <-- EDIT ME: list of (L, hz) cuts to compare

_ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
_SRC = {  # observable -> (dir, value_key, err_key)
    "O_FM": (f"{_ROOT}/xz_line_fm",     "O",   "Oe"),
    "S2":   (f"{_ROOT}/xz_line_s2",     "S2",  "S2e"),
    "B_p":  (f"{_ROOT}/xz_line_energy", "B_p", None),
    "A_v":  (f"{_ROOT}/xz_line_energy", "A_v", None),
    "M_x":  (f"{_ROOT}/xz_line_energy", "mx",  None),
}
_RE = re.compile(r"_L(\d+)_hz(\d+(?:\.\d+)?)")

def _get(obs, L, hz):
    """(hx, y, ye) for one (obs, L, hz), matching hz numerically from the filename; None if absent."""
    d, vk, ek = _SRC[obs]
    for jp in glob.glob(f"{d}/*_L{L}_hz*.json"):
        m = _RE.search(os.path.basename(jp))
        if not m or int(m.group(1)) != L or abs(float(m.group(2)) - hz) > 1e-6:
            continue
        j = json.load(open(jp)); hx = np.array(j["field"], float); o = np.argsort(hx)
        y = np.array(j[vk], float)[o]
        ye = np.array(j[ek], float)[o] if (ek and ek in j) else np.zeros_like(y)
        return hx[o], y, ye
    return None

PANELS = [("O_FM", "O_FM  (string / membrane order param)", None),
          ("S2",   "S₂  (Rényi-2, central plaquette)",       3*np.log(2)),
          ("B_p",  "stabilizers  (⟨B_p⟩ solid, ⟨A_v⟩ dashed)", None),
          ("M_x",  "magnetization  ⟨M_x⟩",                    0.5)]
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
for (obs, title, href), a in zip(PANELS, ax.flat):
    for k, (L, hz) in enumerate(CUTS):
        c = colors[k % 10]
        r = _get(obs, L, hz)
        if r is not None:
            a.errorbar(r[0], r[1], yerr=r[2], fmt="-o", ms=4, capsize=2, color=c,
                       label=f"L={L}, hz={hz}")
        if obs == "B_p":                                   # overlay A_v (pinned ~1 across the hx sweep)
            ra = _get("A_v", L, hz)
            if ra is not None:
                a.plot(ra[0], ra[1], "--", lw=1, color=c, alpha=0.6)
    if href is not None:
        a.axhline(href, ls=":", c="grey", lw=0.8)
    a.set(xlabel="$h_x$", ylabel=obs, title=title)
    a.legend(fontsize=8)
plt.suptitle("XZ magnetic line — observables vs $h_x$ for selected cuts", y=1.01)
plt.tight_layout()
plt.show()